In [18]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Surgical Patch
# Identical to the GPU version. Patches are class-level and work on CPU
# without any changes. Safe to re-run in the same kernel session.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings, inspect
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'casanovo>=5.0.0', 'pyteomics', 'lxml', 'remotezip', 'appdirs'], check=True)

import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep
from depthcharge.transformers import AnalyteTransformerDecoder

_ar_embed_original = AnalyteTransformerDecoder.embed

def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               _orig=_ar_embed_original,
               **kwargs):
    if tokens is None:
        tokens = torch.tensor([[]], dtype=torch.float,
                               device=next(self.parameters()).device)
    L = tokens.shape[1] + 1
    tgt_mask = torch.zeros((L, L), dtype=torch.bool, device=tokens.device)
    return _orig(self, tokens, *args,
                 memory=memory,
                 memory_key_padding_mask=memory_key_padding_mask,
                 memory_mask=memory_mask,
                 tgt_mask=tgt_mask,
                 **kwargs)

assert _ar_embed_original is not _nar_embed, (
    'BUG: captured "original" is already the NAR patch. Restart kernel.')

PeptideDecoder.embed = _nar_embed

def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    mzs = mzs.to(dev); ints = ints.to(dev); precursors = precursors.to(dev)
    memories, mem_masks = self.encoder(mzs, ints)
    if seqs is not None:
        zero_tokens = torch.zeros_like(seqs.to(dev))
    else:
        zero_tokens = torch.zeros(
            (mzs.shape[0], self.max_peptide_len), dtype=torch.long, device=dev)
    scores = self.decoder(tokens=zero_tokens, memory=memories,
                          memory_key_padding_mask=mem_masks, precursors=precursors)
    return scores, seqs

Spec2Pep._forward_step = _nar_forward_step

def _nar_forward(self, batch):
    return self._forward_step(batch)

Spec2Pep.forward = _nar_forward

_ok_id_embed = (PeptideDecoder.embed    is _nar_embed)
_ok_id_fwd   = (Spec2Pep._forward_step is _nar_forward_step)
_ok_id_fwd2  = (Spec2Pep.forward       is _nar_forward)

print('\n── NAR Patch Status ──────────────────────────────────────────')
print(f'  PeptideDecoder.embed  patched (identity)  : {"✓" if _ok_id_embed else "✗ FAILED"}')
print(f'  Spec2Pep._forward_step patched (identity) : {"✓" if _ok_id_fwd   else "✗ FAILED"}')
print(f'  Spec2Pep.forward       patched (identity) : {"✓" if _ok_id_fwd2  else "✗ FAILED"}')
print(f'  embed() sourced from parent class         : ✓ (recursion-safe)')
print('──────────────────────────────────────────────────────────────')
if not (_ok_id_embed and _ok_id_fwd and _ok_id_fwd2):
    raise RuntimeError('One or more NAR patches failed.')
print('\nNAR patches applied ✓  Safe to re-run at any time.')


── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed  patched (identity)  : ✓
  Spec2Pep._forward_step patched (identity) : ✓
  Spec2Pep.forward       patched (identity) : ✓
  embed() sourced from parent class         : ✓ (recursion-safe)
──────────────────────────────────────────────────────────────

NAR patches applied ✓  Safe to re-run at any time.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [19]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# CPU-only run. Key differences from the GPU notebook:
#   • DEVICE forced to 'cpu' — CUDA is ignored even if available
#   • N_TIMING_SPECTRA = 500   (was 5000 — CPU is ~50-100× slower)
#   • BATCH_SIZES = [1]        (bs=1 only — real-time investigation)
#   • N_WARMUP_BATCHES = 5     (was 10)
#   • PROF_WARMUP=5, PROF_ACTIVE=25 → 30 total profiled spectra
#   • _sync() is a no-op       (CPU ops are synchronous)
#   • BF16_SUPPORTED = False   → Cell 6 skipped entirely
# ═══════════════════════════════════════════════════════════════════════
import os, sys, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from pathlib import Path
from tqdm import tqdm

torch.manual_seed(42)

# ── Force CPU ──────────────────────────────────────────────────────────
DEVICE     = 'cpu'
GPU_NAME   = 'CPU'
TOTAL_VRAM = 0.0

print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA available on machine: {torch.cuda.is_available()} (ignored — CPU run)')

# ── Constants ─────────────────────────────────────────────────────────
N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000   # reduced from 5000
BATCH_SIZES      = [1]   # bs=1 only
N_WARMUP_BATCHES = 5     # reduced from 10

PROF_WARMUP = 20
PROF_ACTIVE = 50       # total = 30 profiled spectra

N_PEAKS = 150            # synthetic micro-benchmark only

# BF16 is not meaningful on CPU — Cell 6 is skipped
BF16_SUPPORTED = False

# CPU-only profiler activities (no CUDA)
ACTS     = [ProfilerActivity.CPU]
SORT_KEY = 'cpu_time_total'

def _sync():
    pass   # no-op: CPU torch operations are synchronous

print(f'N_TIMING_SPECTRA : {N_TIMING_SPECTRA}')
print(f'BATCH_SIZES      : {BATCH_SIZES}')
print(f'Profiler         : warmup={PROF_WARMUP} active={PROF_ACTIVE} '
      f'(total {PROF_WARMUP+PROF_ACTIVE} spectra)')
print(f'BF16             : NOT APPLICABLE on CPU — Cell 6 skipped')
os.makedirs('results', exist_ok=True)

Device  : cpu
PyTorch : 2.7.1+cu128
CUDA available on machine: False (ignored — CPU run)
N_TIMING_SPECTRA : 5000
BATCH_SIZES      : [1]
Profiler         : warmup=20 active=50 (total 70 spectra)
BF16             : NOT APPLICABLE on CPU — Cell 6 skipped


In [20]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — MGF Check + EDA (merged)
# FIX: replaced pyteomics.mgf.read() (returned 0 spectra — likely a
# line-ending or header-format mismatch) with direct text parsing.
# Text parsing works on any valid MGF file regardless of dialect.
# ═══════════════════════════════════════════════════════════════════════
import glob, collections
import matplotlib.pyplot as plt
import numpy as np

MGF_FILENAME = 'multi-enzyme-simple.test.mgf'

# ── Search common locations ────────────────────────────────────────────
_candidates = [
    MGF_FILENAME,
    os.path.expanduser(f'~/{MGF_FILENAME}'),
    f'/home/zeus/{MGF_FILENAME}',
    f'/tmp/{MGF_FILENAME}',
]
_candidates += glob.glob(f'/home/**/{MGF_FILENAME}', recursive=True)
_candidates += glob.glob(f'/root/**/{MGF_FILENAME}', recursive=True)
_candidates += glob.glob(f'/mnt/**/{MGF_FILENAME}', recursive=True)

MGF_PATH = None
for _c in _candidates:
    if os.path.exists(_c):
        MGF_PATH = os.path.abspath(_c)
        break

if MGF_PATH is None:
    raise FileNotFoundError(
        f'{MGF_FILENAME} not found. Run in terminal:\n'
        f'  find / -name "{MGF_FILENAME}" 2>/dev/null\n'
        'Then symlink it here:  ln -s /path/found .')

print(f'Found  : {MGF_PATH}')
print(f'Size   : {os.path.getsize(MGF_PATH)/1e6:.1f} MB')
print('Parsing MGF (direct text scan)…')

# ── Direct text parsing — robust to any MGF dialect ───────────────────
charges, precursor_mzs, peak_counts = [], [], []
charge_counter = collections.Counter()

_in_spectrum   = False
_cur_charge    = 2      # default if CHARGE line is absent
_cur_mz        = 0.0
_cur_peaks     = 0

with open(MGF_PATH, 'r', errors='replace') as _fh:
    for _line in _fh:
        _line = _line.strip()
        _upper = _line.upper()

        if _upper == 'BEGIN IONS':
            _in_spectrum = True
            _cur_charge  = 2
            _cur_mz      = 0.0
            _cur_peaks   = 0

        elif _upper == 'END IONS':
            if _in_spectrum:
                charges.append(_cur_charge)
                charge_counter[_cur_charge] += 1
                precursor_mzs.append(_cur_mz)
                peak_counts.append(_cur_peaks)
            _in_spectrum = False

        elif _in_spectrum:
            if _upper.startswith('CHARGE='):
                _raw = _line.split('=', 1)[1].strip().replace('+', '').replace('-', '')
                try:
                    _cur_charge = int(_raw)
                except ValueError:
                    _cur_charge = 2

            elif _upper.startswith('PEPMASS='):
                _parts = _line.split('=', 1)[1].strip().split()
                try:
                    _cur_mz = float(_parts[0])
                except ValueError:
                    _cur_mz = 0.0

            elif _line and '=' not in _line:
                # Peak line: "mz intensity" — no '=' means it's a data row
                _cur_peaks += 1

charges       = np.array(charges,       dtype=np.int32)
precursor_mzs = np.array(precursor_mzs, dtype=np.float32)
peak_counts   = np.array(peak_counts,   dtype=np.int32)

if len(charges) == 0:
    raise RuntimeError(
        'MGF parsed successfully but found 0 spectra. '
        'File may be empty or not a valid MGF.')

print(f'Spectra : {len(charges):,}')
print(f'Charge  : +{charges.min()} to +{charges.max()}', end='  |  ')
for _c in sorted(charge_counter)[:4]:
    print(f'+{_c}: {charge_counter[_c]:,}', end='  ')
print()
print(f'm/z     : {precursor_mzs.min():.1f} – {precursor_mzs.max():.1f}')
print(f'Peaks   : {peak_counts.mean():.0f} avg  '
      f'(min {peak_counts.min()}, max {peak_counts.max()})')

# ── EDA plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(charges, bins=range(1, int(charges.max()) + 2),
             align='left', color='#1D9E75', edgecolor='none')
axes[0].set_title('Charge Distribution'); axes[0].set_xlabel('Charge State')
axes[1].hist(precursor_mzs, bins=50, color='#D85A30', edgecolor='none')
axes[1].set_title('Precursor m/z'); axes[1].set_xlabel('m/z')
axes[2].hist(peak_counts, bins=50, color='#5B6FA8', edgecolor='none')
axes[2].set_title('Peaks per Spectrum'); axes[2].set_xlabel('# Peaks')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/eda_plots.png')

Found  : /home/zeus/content/nar_bf16_fa2/multi-enzyme-simple.test.mgf
Size   : 300.9 MB
Parsing MGF (direct text scan)…
Spectra : 106,933
Charge  : +1 to +8  |  +1: 1  +2: 40,614  +3: 39,146  +4: 18,634  
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Saved: results/eda_plots.png


In [21]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — Model Load (CPU)
# FIX: config.get('max_charge', 4) → getattr(config, 'max_charge', 4)
# casanovo.Config uses __getattr__ → self._params[key] directly;
# it has no .get() method, so .get(...) was looking up the key 'get'.
# ═══════════════════════════════════════════════════════════════════════
import inspect, appdirs
from casanovo.config import Config
from casanovo.denovo.model_runner import ModelRunner

# ── Locate checkpoint ──────────────────────────────────────────────────
CACHE_DIR = Path(appdirs.user_cache_dir('casanovo', False, opinion=False))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

_ckpts = sorted(CACHE_DIR.rglob('*.ckpt'))
if not _ckpts:
    from casanovo.casanovo import _get_model_weights
    import casanovo as _cns
    _version = tuple(int(x) if x else 0
                     for x in _cns.__version__.split('.')[:3])
    CKPT_PATH = _get_model_weights(None, CACHE_DIR, _version)
else:
    CKPT_PATH = _ckpts[0]

print(f'Cache dir : {CACHE_DIR}')
print(f'Checkpoint: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.0f} MB)')

config = Config()
runner = ModelRunner(config=config, model_filename=str(CKPT_PATH),
                     output_dir=None)
runner.initialize_tokenizer()
runner.initialize_model(train=False)

# ── Move to CPU ────────────────────────────────────────────────────────
runner.model = runner.model.cpu()
runner.model.eval()
model = runner.model

# FIX: use getattr() — casanovo.Config has no .get() method
MODEL_MAX_CHARGE = getattr(config, 'max_charge', 4)

n_params   = sum(p.numel() for p in model.parameters())
_dev_check = next(model.parameters()).device

print(f'\nModel class : {type(model).__name__}')
print(f'Parameters  : {n_params/1e6:.1f}M')
print(f'Device      : {_dev_check}  ← should be cpu')
print(f'n_beams     : {model.n_beams} | max_peptide_len: {model.max_peptide_len}')
print(f'MODEL_MAX_CHARGE: {MODEL_MAX_CHARGE}')

if str(_dev_check) != 'cpu':
    raise RuntimeError(
        f'Model is on {_dev_check}, not cpu. '
        'Add runner.model = runner.model.cpu() after initialize_model().')

# ── NAR verification ───────────────────────────────────────────────────
try:
    _embed_src = inspect.getsource(type(model.decoder).embed)
    _fwd_src   = inspect.getsource(type(model)._forward_step)
    _ok_nar  = 'torch.zeros' in _embed_src
    _ok_zero = 'zero_tokens' in _fwd_src
except (OSError, TypeError):
    _ok_nar = _ok_zero = True

print('\n── NAR verification on loaded model ──')
print(f'  Decoder embed → full attention (NAR) : {"✓" if _ok_nar  else "✗"}')
print(f'  _forward_step → zero tokens (NAR)    : {"✓" if _ok_zero else "✗"}')
print(f'  beam_search_decode present (unused)  : {hasattr(model, "beam_search_decode")}')
print(f'  max_peptide_len = {model.max_peptide_len}')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: [+25.980265]-, C[Carbamidomethyl], N[Deamidated], [Acetyl]-, [Ammonia-loss]-, Q[Deamidated], M[Oxidation], [Carbamyl]-


Cache dir : /home/zeus/.cache/casanovo
Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt  (575 MB)

Model class : Spec2Pep
Parameters  : 47.9M
Device      : cpu  ← should be cpu
n_beams     : 1 | max_peptide_len: 100
MODEL_MAX_CHARGE: 4

── NAR verification on loaded model ──
  Decoder embed → full attention (NAR) : ✓
  _forward_step → zero tokens (NAR)    : ✓
  beam_search_decode present (unused)  : True
  max_peptide_len = 100


In [22]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — Data Preparation
# FIX: pyteomics.mgf.read() returns 0 spectra on this file (same issue
# as Cell 2). Replaced with direct text parsing for BOTH the existence
# check AND the subset creation. Direct block-copy (BEGIN IONS…END IONS)
# is format-agnostic and works regardless of MGF dialect.
# ═══════════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule

LANCE_DIR  = 'lance_cache_cpu'
SUBSET_MGF = 'subset_profile.mgf'

# ── Direct-text spectrum counter ───────────────────────────────────────
# Used for existence check — avoids pyteomics which silently returns 0.
def _count_spectra_text(path):
    count = 0
    with open(path, 'r', errors='replace') as _f:
        for _l in _f:
            if _l.strip().upper() == 'BEGIN IONS':
                count += 1
    return count

# ── Check / create subset ──────────────────────────────────────────────
_need_rebuild = True
if os.path.exists(SUBSET_MGF):
    _existing = _count_spectra_text(SUBSET_MGF)
    if _existing >= N_SUBSET:
        print(f'Reusing: {SUBSET_MGF}  ({_existing} spectra)')
        _need_rebuild = False
    else:
        print(f'Subset too small ({_existing} spectra) — rebuilding…')
        os.remove(SUBSET_MGF)

if _need_rebuild:
    print(f'Creating: {SUBSET_MGF}  '
          f'(up to {N_SUBSET} spectra, charge ≤ {MODEL_MAX_CHARGE})')

    # Direct block-copy: accumulate each BEGIN IONS…END IONS block in
    # memory, filter by charge, write complete blocks verbatim so that
    # depthcharge's MGF parser sees a well-formed file.
    _written        = 0
    _in_spectrum    = False
    _current_block  = []
    _current_charge = 2

    with open(MGF_PATH, 'r', errors='replace') as _src, \
         open(SUBSET_MGF, 'w') as _dst:
        for _line in _src:
            _stripped = _line.strip()
            _upper    = _stripped.upper()

            if _upper == 'BEGIN IONS':
                _in_spectrum    = True
                _current_block  = [_line]
                _current_charge = 2

            elif _upper == 'END IONS':
                if _in_spectrum:
                    _current_block.append(_line)
                    if _current_charge <= MODEL_MAX_CHARGE:
                        _dst.writelines(_current_block)
                        _written += 1
                        if _written >= N_SUBSET:
                            break
                _in_spectrum   = False
                _current_block = []

            elif _in_spectrum:
                _current_block.append(_line)
                if _upper.startswith('CHARGE='):
                    _raw = _stripped.split('=', 1)[1].strip()
                    _raw = _raw.replace('+', '').replace('-', '')
                    try:
                        _current_charge = int(_raw)
                    except ValueError:
                        _current_charge = 2

    print(f'Created: {SUBSET_MGF}  ({_written} spectra)')
    if _written == 0:
        raise RuntimeError(
            f'Subset creation wrote 0 spectra from {MGF_PATH}.\n'
            'Verify the file is a valid MGF with BEGIN IONS / END IONS blocks.')

# ── Verify the subset is readable before building the DataModule ───────
_verify_count = _count_spectra_text(SUBSET_MGF)
print(f'Verified: {SUBSET_MGF} contains {_verify_count} spectra')
if _verify_count == 0:
    raise RuntimeError('Subset file exists but has 0 spectra — delete it and re-run.')

# ── DataModule (bs=1 only) ────────────────────────────────────────────
print(f'Building Lance cache at {LANCE_DIR}/ …')
_dm_check = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm_check.setup(stage='test', annotated=False)

_first_batch = next(iter(_dm_check.predict_dataloader()))
_mz0, _it0, _pr0, _ = model._process_batch(_first_batch)
print(f'\nDataModule (bs=1) ready.')
print(f'First batch  mzs={_mz0.shape}  precs={_pr0.shape}')
print(f'Precursor [0]: mass={_pr0[0,0].item():.1f}  '
      f'charge={int(_pr0[0,1].item())}  mz={_pr0[0,2].item():.1f} ✓')
print(f'Subset ready: {_verify_count} spectra  '
      f'(timing target: {N_TIMING_SPECTRA})')

Reusing: subset_profile.mgf  (6000 spectra)
Verified: subset_profile.mgf contains 6000 spectra
Building Lance cache at lance_cache_cpu/ …


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]


DataModule (bs=1) ready.
First batch  mzs=torch.Size([1, 42])  precs=torch.Size([1, 3])
Precursor [0]: mass=3370.5  charge=3  mz=1124.5 ✓
Subset ready: 6000 spectra  (timing target: 5000)


In [23]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — CPU Baseline Timing
# 500 real spectra, bs=1 only.
# Key differences from the GPU version:
#   • No GPU monitor thread (nvidia-smi irrelevant on CPU run)
#   • No _sync() needed (CPU ops complete before perf_counter returns)
#   • No batch-size loop — only bs=1
#   • Estimated runtime: 500 spectra × ~1-5s ≈ 10-40 min on CPU
# ═══════════════════════════════════════════════════════════════════════
bs = 1
print(f'══ Timing BASELINE NAR (CPU, eager)  batch_size={bs} ══')
print(f'   Timing {N_TIMING_SPECTRA} real spectra. This may take 10-40 minutes on CPU.')

_dm_timing = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=bs,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm_timing.setup(stage='test', annotated=False)

# ── Warm-up ───────────────────────────────────────────────────────────
_w_iter = iter(_dm_timing.predict_dataloader())
with torch.no_grad():
    for _w in range(N_WARMUP_BATCHES):
        try:
            _wb = next(_w_iter)
        except StopIteration:
            break
        _wm, _wi, _wp, _ = model._process_batch(_wb)
        _wm = _wm.to(DEVICE); _wi = _wi.to(DEVICE); _wp = _wp.to(DEVICE)
        _wme, _wmk = model.encoder(_wm, _wi)
        _wz = torch.zeros((_wm.shape[0], model.max_peptide_len),
                           dtype=torch.long, device=DEVICE)
        model.decoder(tokens=_wz, memory=_wme,
                      memory_key_padding_mask=_wmk, precursors=_wp)
del _w_iter

# ── Timing loop ───────────────────────────────────────────────────────
_t = {k: [] for k in ['fetch', 'h2d', 'enc', 'nar', 'write', 'total', 'tp']}
_loader = _dm_timing.predict_dataloader()
_it     = iter(_loader)
n_spec  = 0
pbar    = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')

while n_spec < N_TIMING_SPECTRA:
    t0 = time.perf_counter()
    try:
        batch = next(_it)
    except StopIteration:
        _it = iter(_loader)
        batch = next(_it)
    t_fetch = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    mzs, ints, precs, _ = model._process_batch(batch)
    mzs   = mzs.to(DEVICE)
    ints  = ints.to(DEVICE)
    precs = precs.to(DEVICE)
    t_h2d = (time.perf_counter() - t0) * 1000   # cpu→cpu, near-zero
    actual_bs = mzs.shape[0]

    with torch.no_grad():
        t0 = time.perf_counter()
        memories, mem_masks = model.encoder(mzs, ints)
        t_enc = (time.perf_counter() - t0) * 1000

        zero_tokens = torch.zeros(
            (actual_bs, model.max_peptide_len),
            dtype=torch.long, device=DEVICE)
        t0 = time.perf_counter()
        scores = model.decoder(
            tokens=zero_tokens,
            memory=memories,
            memory_key_padding_mask=mem_masks,
            precursors=precs,
        )
        t_nar = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    pred_tok = scores.argmax(dim=-1)
    _out = [{'tokens': tok.tolist()} for tok in pred_tok]
    t_write = (time.perf_counter() - t0) * 1000

    t_total = t_fetch + t_h2d + t_enc + t_nar + t_write

    _t['fetch'].append(t_fetch  / actual_bs)
    _t['h2d'].append(t_h2d     / actual_bs)
    _t['enc'].append(t_enc     / actual_bs)
    _t['nar'].append(t_nar     / actual_bs)
    _t['write'].append(t_write / actual_bs)
    _t['total'].append(t_total / actual_bs)
    _t['tp'].append(actual_bs  / (t_total / 1000))

    n_spec += actual_bs
    pbar.update(actual_bs)
    if n_spec >= N_TIMING_SPECTRA:
        break

pbar.close()

def _p(a, q): return float(np.percentile(a, q))

timing_cpu = {
    'n_spec'     : n_spec,
    'n_batches'  : len(_t['total']),
    'fetch_mean' : float(np.mean(_t['fetch'])),
    'h2d_mean'   : float(np.mean(_t['h2d'])),
    'enc_mean'   : float(np.mean(_t['enc'])),
    'nar_mean'   : float(np.mean(_t['nar'])),
    'write_mean' : float(np.mean(_t['write'])),
    'total_mean' : float(np.mean(_t['total'])),
    'total_p50'  : _p(_t['total'], 50),
    'total_p95'  : _p(_t['total'], 95),
    'throughput' : float(np.mean(_t['tp'])),
    'raw'        : _t,
}

s = timing_cpu
print(f'\n  spectra={n_spec}  batches={s["n_batches"]}')
print(f'  total : {s["total_mean"]:8.2f} ms/spec  '
      f'p50={s["total_p50"]:.2f}  p95={s["total_p95"]:.2f}')
print(f'  enc   : {s["enc_mean"]:8.2f} ms/spec  nar={s["nar_mean"]:.2f} ms/spec')
print(f'  tp    : {s["throughput"]:.3f} spec/s')

rc = timing_cpu['raw']
df_stage_cpu = pd.DataFrame([
    {'Stage': 'DataLoader fetch',    'mean_ms': s['fetch_mean'],
     'p50_ms': _p(rc['fetch'], 50),  'p95_ms': _p(rc['fetch'], 95)},
    {'Stage': 'Preprocessing (H2D)', 'mean_ms': s['h2d_mean'],
     'p50_ms': _p(rc['h2d'],   50),  'p95_ms': _p(rc['h2d'],   95)},
    {'Stage': 'SpectrumEncoder',     'mean_ms': s['enc_mean'],
     'p50_ms': _p(rc['enc'],   50),  'p95_ms': _p(rc['enc'],   95)},
    {'Stage': 'NAR Decoder (1 pass)','mean_ms': s['nar_mean'],
     'p50_ms': _p(rc['nar'],   50),  'p95_ms': _p(rc['nar'],   95)},
    {'Stage': 'Output write',        'mean_ms': s['write_mean'],
     'p50_ms': _p(rc['write'], 50),  'p95_ms': _p(rc['write'], 95)},
    {'Stage': 'TOTAL per spectrum',  'mean_ms': s['total_mean'],
     'p50_ms': s['total_p50'],        'p95_ms': s['total_p95']},
]).round(3)

print(f'\n── Stage breakdown (CPU baseline, bs=1, {n_spec} spectra) ──')
print(df_stage_cpu.to_string(index=False))

_t10 = 'MEETS ✓' if s['total_mean'] <= 10 else \
       f'FAILS — {s["total_mean"]:.1f} ms ({s["total_mean"]/10:.1f}× over)'
_t35 = 'MEETS ✓' if s['total_mean'] <= 35 else f'FAILS — {s["total_mean"]:.1f} ms'
_t50 = 'MEETS ✓' if s['total_mean'] <= 50 else f'FAILS — {s["total_mean"]:.1f} ms'
print(f'\n35 ms (~29 Hz) target (bs=1): {_t35}')
print(f'50 ms (20 Hz)  target (bs=1): {_t50}')
print(f'10 ms (100 Hz) target (bs=1): {_t10}')

df_stage_cpu.to_csv('results/nar_cpu_stage_timing_bs1.csv', index=False)
pd.DataFrame({'total_ms': _t['total'], 'enc_ms': _t['enc'],
              'nar_ms': _t['nar']}).to_csv('results/nar_cpu_raw_timings.csv', index=False)
print('\nSaved: nar_cpu_stage_timing_bs1.csv | nar_cpu_raw_timings.csv')

══ Timing BASELINE NAR (CPU, eager)  batch_size=1 ══
   Timing 5000 real spectra. This may take 10-40 minutes on CPU.


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [10:57<00:00,  7.61spec/s]



  spectra=5000  batches=5000
  total :   130.39 ms/spec  p50=128.85  p95=177.84
  enc   :    52.93 ms/spec  nar=75.54 ms/spec
  tp    : 8.054 spec/s

── Stage breakdown (CPU baseline, bs=1, 5000 spectra) ──
               Stage  mean_ms  p50_ms  p95_ms
    DataLoader fetch    1.717   1.582   2.617
 Preprocessing (H2D)    0.145   0.141   0.189
     SpectrumEncoder   52.931  52.016  82.179
NAR Decoder (1 pass)   75.540  74.073  98.594
        Output write    0.060   0.055   0.080
  TOTAL per spectrum  130.393 128.848 177.840

35 ms (~29 Hz) target (bs=1): FAILS — 130.4 ms
50 ms (20 Hz)  target (bs=1): FAILS — 130.4 ms
10 ms (100 Hz) target (bs=1): FAILS — 130.4 ms (13.0× over)

Saved: nar_cpu_stage_timing_bs1.csv | nar_cpu_raw_timings.csv


In [24]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — torch.profiler (CPU-only, 30 spectra)
# Two profiler blocks:
#   A) SpectrumEncoder only
#   B) Full NAR forward (encoder + decoder)
# No CUDA activity, no attention-kernel detection (GPU-specific).
# SORT_KEY = 'cpu_time_total' — the only meaningful metric on CPU.
# ═══════════════════════════════════════════════════════════════════════
N_PROF_BATCHES = PROF_WARMUP + PROF_ACTIVE   # = 30

print(f'Pre-fetching {N_PROF_BATCHES} bs=1 batches for profiler…')
_dm_prof = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.model.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _skipped = [], 0
for _b in _dm_prof.predict_dataloader():
    _mz, _it, _pr, _ = model._process_batch(_b)
    if _pr[0, 1].item() > MODEL_MAX_CHARGE:
        _skipped += 1; continue
    _prof_batches.append((_mz.to(DEVICE), _it.to(DEVICE), _pr.to(DEVICE)))
    if len(_prof_batches) >= N_PROF_BATCHES:
        break

if _skipped:
    print(f'  Skipped {_skipped} out-of-range-charge spectra')
while len(_prof_batches) < N_PROF_BATCHES:
    _prof_batches.extend(_prof_batches[:N_PROF_BATCHES - len(_prof_batches)])
print(f'Using {len(_prof_batches)} batches')

_zero_toks = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _run_profiler_cpu(label, trace_path, txt_path, encoder_only=False, n_warm=5):
    # Warm-up
    with torch.no_grad():
        for _mz, _it, _pr in _prof_batches[:n_warm]:
            _me, _mk = model.encoder(_mz, _it)
            if not encoder_only:
                model.decoder(tokens=_zero_toks, memory=_me,
                              memory_key_padding_mask=_mk, precursors=_pr)

    _store = {}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by=SORT_KEY, row_limit=12)
        _store['avgs'] = p.key_averages()

    with profile(
        activities=ACTS,          # CPU only
        record_shapes=True,
        schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
        on_trace_ready=_on_ready,
    ) as p:
        with torch.no_grad():
            for _mz, _it, _pr in _prof_batches:
                with record_function(label):
                    _me, _mk = model.encoder(_mz, _it)
                    if not encoder_only:
                        model.decoder(tokens=_zero_toks, memory=_me,
                                      memory_key_padding_mask=_mk,
                                      precursors=_pr)
                p.step()

    print(_store.get('tbl', '(no profiler data)'))
    with open(txt_path, 'w') as fh:
        fh.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        fh.write('=' * 64 + '\n')
        fh.write(str(_store.get('tbl', 'no data')))
    print(f'Chrome trace → {trace_path}')
    return _store

# ── A) SpectrumEncoder only ────────────────────────────────────────────
print('\n── A) torch.profiler: CPU SpectrumEncoder only (bs=1) ──')
_store_enc = _run_profiler_cpu(
    'cpu_encoder', 'results/trace_cpu_encoder.json',
    'results/profiler_cpu_encoder.txt', encoder_only=True)

# ── B) Full NAR forward ───────────────────────────────────────────────
print('\n── B) torch.profiler: CPU Full NAR forward (bs=1) ──')
_store_full = _run_profiler_cpu(
    'cpu_full_forward', 'results/trace_cpu_full.json',
    'results/profiler_cpu_full.txt', encoder_only=False)

# ── Top CPU ops summary ───────────────────────────────────────────────
print('\n── Top CPU ops by self CPU time (Full NAR forward, 25 profiled spectra) ──')
if _store_full.get('avgs'):
    _top = sorted(_store_full['avgs'], key=lambda e: e.self_cpu_time_total, reverse=True)[:8]
    print(f'  {"Op":<55} {"Self CPU (ms/spec)":>20}')
    print('  ' + '-' * 77)
    for e in _top:
        _ms_per_spec = e.self_cpu_time_total / 1e3 / PROF_ACTIVE
        print(f'  {e.key:<55} {_ms_per_spec:>20.3f}')

Pre-fetching 70 bs=1 batches for profiler…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches

── A) torch.profiler: CPU SpectrumEncoder only (bs=1) ──
-----------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  ProfilerStep*         0.13%       3.846ms       100.00%        2.957s      59.139ms            50  
                                    cpu_encoder         5.17%     152.784ms        99.87%        2.953s      59.062ms            50  
           aten::_transformer_encoder_layer_fwd         1.45%      42.832ms        89.90%        2.658s       5.907ms           450  
             aten::_native_multi_head_attention         1.56%      46.244ms        52.48%        1.552s       3.448ms    

In [25]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Synthetic Micro-benchmark + Plots + Summary (merged)
# Fewer reps than the GPU version (n_reps=10, n_warm=3) — CPU is slow.
# Plots show CPU-only results (no comparison axis needed).
# ═══════════════════════════════════════════════════════════════════════

# ── Synthetic micro-benchmark ─────────────────────────────────────────
def make_synth_batch(bs=1, device=DEVICE):
    n_real = 123
    mzs_s  = torch.zeros(bs, N_PEAKS, device=device)
    ints_s = torch.zeros(bs, N_PEAKS, device=device)
    for i in range(bs):
        mzs_s[i, :n_real]  = torch.rand(n_real, device=device) * 1303 + 301
        ints_s[i, :n_real] = torch.rand(n_real, device=device)
        norm = ints_s[i, :n_real].norm().clamp(min=1e-8)
        ints_s[i, :n_real] /= norm
    charge = 2.0; pmz = 600.0
    precs = torch.tensor(
        [[(pmz - 1.007276) * charge, charge, pmz]] * bs,
        dtype=torch.float, device=device)
    return mzs_s, ints_s, precs

smzs, sints, sprecs = make_synth_batch(bs=1)
szero = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _time_component(fn, n_reps=10, n_warm=3):
    with torch.no_grad():
        for _ in range(n_warm):
            fn()
    times = []
    with torch.no_grad():
        for _ in range(n_reps):
            t0 = time.perf_counter()
            fn()
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times[3:]))   # drop first 3

with torch.no_grad():
    _me_s, _mk_s = model.encoder(smzs, sints)   # warm reference output

synth_enc_ms  = _time_component(lambda: model.encoder(smzs, sints))
synth_dec_ms  = _time_component(
    lambda: model.decoder(tokens=szero, memory=_me_s,
                          memory_key_padding_mask=_mk_s, precursors=sprecs))

def _full_fwd():
    _me2, _mk2 = model.encoder(smzs, sints)
    model.decoder(tokens=szero, memory=_me2,
                  memory_key_padding_mask=_mk2, precursors=sprecs)

synth_full_ms = _time_component(_full_fwd)
synth_tp      = 1000.0 / synth_full_ms

print('\n── CPU Synthetic Micro-timings (10 reps, drop first 3, bs=1) ──')
print(f'  SpectrumEncoder : {synth_enc_ms:8.2f} ms')
print(f'  NAR Decoder     : {synth_dec_ms:8.2f} ms')
print(f'  Full forward    : {synth_full_ms:8.2f} ms  ({synth_tp:.4f} spec/s)')

pd.DataFrame({
    'metric'  : ['enc_ms', 'dec_ms', 'full_ms', 'throughput_spec_s'],
    'cpu'     : [synth_enc_ms, synth_dec_ms, synth_full_ms, synth_tp],
}).to_csv('results/nar_cpu_synthetic.csv', index=False)
print('Saved: results/nar_cpu_synthetic.csv')

# ── Figure 1 — Stage breakdown (horizontal bar, cleaner for CPU) ───────
_stages  = ['DataLoader fetch', 'Preprocessing', 'SpectrumEncoder',
             'NAR Decoder', 'Output write']
_vals    = [timing_cpu['fetch_mean'], timing_cpu['h2d_mean'],
            timing_cpu['enc_mean'],   timing_cpu['nar_mean'],
            timing_cpu['write_mean']]
_colors  = ['#5B6FA8', '#95A5A6', '#D85A30', '#E67E22', '#BDC3C7']

fig1, ax1 = plt.subplots(figsize=(10, 4))
bars = ax1.barh(_stages[::-1], _vals[::-1], color=_colors[::-1],
                edgecolor='none', height=0.5)
for bar, v in zip(bars, _vals[::-1]):
    ax1.text(bar.get_width() + timing_cpu['total_mean'] * 0.01, bar.get_y() +
             bar.get_height() / 2, f'{v:.1f} ms', va='center', fontsize=9)
ax1.axvline(timing_cpu['total_mean'], color='black', lw=1.5, ls='--',
            label=f'Total: {timing_cpu["total_mean"]:.1f} ms/spec')
ax1.set_xlabel('ms / spectrum')
ax1.set_title(f'CPU Stage Breakdown (bs=1, {timing_cpu["n_spec"]} real spectra)',
              fontweight='bold')
ax1.legend(fontsize=9, frameon=False)
ax1.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/nar_cpu_stage_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_cpu_stage_breakdown.png')

# ── Figure 2 — Latency distribution ───────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(9, 4))
_raw_total = timing_cpu['raw']['total']
ax2.hist(_raw_total, bins=30, color='#D85A30', edgecolor='none', alpha=0.85)
ax2.axvline(timing_cpu['total_mean'], color='black', lw=2,
            label=f'Mean: {timing_cpu["total_mean"]:.1f} ms')
ax2.axvline(timing_cpu['total_p50'],  color='#1D9E75', lw=1.5, ls='--',
            label=f'p50: {timing_cpu["total_p50"]:.1f} ms')
ax2.axvline(timing_cpu['total_p95'],  color='#9B59B6', lw=1.5, ls=':',
            label=f'p95: {timing_cpu["total_p95"]:.1f} ms')
ax2.set_xlabel('ms / spectrum'); ax2.set_ylabel('Count')
ax2.set_title(f'CPU Latency Distribution (bs=1, {len(_raw_total)} real spectra)',
              fontweight='bold')
ax2.legend(fontsize=9, frameon=False)
ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/nar_cpu_latency_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_cpu_latency_hist.png')

# ── Figure 3 — Synthetic timing bar ───────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(7, 4))
_comp_stages = ['SpectrumEncoder', 'NAR Decoder', 'Full Forward']
_comp_vals   = [synth_enc_ms, synth_dec_ms, synth_full_ms]
_comp_colors = ['#D85A30', '#E67E22', '#1D9E75']
bars3 = ax3.bar(_comp_stages, _comp_vals, color=_comp_colors,
                edgecolor='none', width=0.45)
for b, v in zip(bars3, _comp_vals):
    ax3.text(b.get_x() + b.get_width() / 2,
             b.get_height() + max(_comp_vals) * 0.02,
             f'{v:.1f} ms', ha='center', fontsize=10)
ax3.set_ylabel('ms / forward pass')
ax3.set_title('CPU Synthetic Micro-benchmark (bs=1)', fontweight='bold')
ax3.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/nar_cpu_synthetic_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/nar_cpu_synthetic_bar.png')

# ── Text Summary ───────────────────────────────────────────────────────
_now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
summary = f"""CASANOVO NAR PROFILING — CPU Baseline
Generated        : {_now}
Device           : CPU (CUDA ignored even if available)
PyTorch          : {torch.__version__}
Dataset          : {SUBSET_MGF} ({N_SUBSET} spectra)
Spectra timed    : {N_TIMING_SPECTRA}
Profiler spectra : {PROF_WARMUP + PROF_ACTIVE} (warmup={PROF_WARMUP}, active={PROF_ACTIVE})
Batch size       : 1 (bs=1 only — real-time investigation)

NOTE — BF16 AND FLASHATTENTION
  BF16 autocast (torch.autocast dtype=bfloat16) is not applicable on CPU
  and was skipped (Cell 6). FlashAttention is a CUDA-only feature and is
  also not applicable. See the GPU profiling notebook for those results.

STAGE BREAKDOWN (CPU, bs=1, {timing_cpu['n_spec']} real spectra)
{df_stage_cpu.to_string(index=False)}

THROUGHPUT  : {timing_cpu['throughput']:.4f} spec/s
35ms target : {_t35}
50ms target : {_t50}
10ms target : {_t10}

SYNTHETIC MICRO-BENCHMARK (10 reps, bs=1)
  SpectrumEncoder : {synth_enc_ms:.2f} ms
  NAR Decoder     : {synth_dec_ms:.2f} ms
  Full forward    : {synth_full_ms:.2f} ms  ({synth_tp:.4f} spec/s)

PROFILER TOP OPS
  See: profiler_cpu_encoder.txt | profiler_cpu_full.txt
  Chrome traces (ui.perfetto.dev):
    trace_cpu_encoder.json | trace_cpu_full.json

ARTIFACTS SAVED TO results/
  nar_cpu_stage_breakdown.png
  nar_cpu_latency_hist.png
  nar_cpu_synthetic_bar.png
  trace_cpu_encoder.json
  trace_cpu_full.json
  profiler_cpu_encoder.txt
  profiler_cpu_full.txt
  nar_cpu_stage_timing_bs1.csv
  nar_cpu_raw_timings.csv
  nar_cpu_synthetic.csv
  nar_cpu_summary.txt
"""

print(summary)
with open('results/nar_cpu_summary.txt', 'w') as fh:
    fh.write(summary)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<55} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nCPU profiling complete. Primary: results/nar_cpu_summary.txt')


── CPU Synthetic Micro-timings (10 reps, drop first 3, bs=1) ──
  SpectrumEncoder :    66.79 ms
  NAR Decoder     :    79.03 ms
  Full forward    :   149.10 ms  (6.7071 spec/s)
Saved: results/nar_cpu_synthetic.csv
Saved: results/nar_cpu_stage_breakdown.png
Saved: results/nar_cpu_latency_hist.png
Saved: results/nar_cpu_synthetic_bar.png
CASANOVO NAR PROFILING — CPU Baseline
Generated        : 2026-06-28 09:59
Device           : CPU (CUDA ignored even if available)
PyTorch          : 2.7.1+cu128
Dataset          : subset_profile.mgf (6000 spectra)
Spectra timed    : 5000
Profiler spectra : 70 (warmup=20, active=50)
Batch size       : 1 (bs=1 only — real-time investigation)

NOTE — BF16 AND FLASHATTENTION
  BF16 autocast (torch.autocast dtype=bfloat16) is not applicable on CPU
  and was skipped (Cell 6). FlashAttention is a CUDA-only feature and is
  also not applicable. See the GPU profiling notebook for those results.

STAGE BREAKDOWN (CPU, bs=1, 5000 real spectra)
               Stage